In [32]:
import os
import argparse
import numpy as np
import pandas as pd
from decimal import *
import matplotlib.pyplot as plt

In [33]:
def update_delta(gamma_j, beta):
    return 1/gamma_j * ((gamma_j*(beta - 1) + 1)**(1/(1 - beta)))

def update_eff(n_a, E_o, S_o, delta_e, k_C, n_d):
    # if E_o - n_a* S_o is positive, the exponential value of this expression will explode in calculation.
    # The original form of efficiency formula contains negative exponent and
    # the simplified form contains the positive exponent
    
    if (E_o - n_a*S_o) >= 0:
        return n_d * (n_a - \
                        ((E_o - n_a*S_o)*n_a*np.exp(-(E_o - n_a * S_o) * k_C * delta_e))/ \
                        (E_o - n_a*S_o*np.exp(-(E_o - n_a*S_o) * k_C * delta_e)))
    else:
        return n_d * (n_a - \
                    ((E_o - n_a*S_o) * n_a)/ \
                        (E_o * np.exp((E_o - n_a * S_o) * k_C * delta_e) - n_a * S_o))
    
def simulate_vals(D_init, E_init, P_init, delta_e, n_d, n_dE, k_C, beta, n_cycle=50):
    S_o, gamma, delta, n_a, eff = [], [], [], [], []
    P_o, E_o, D_e = [P_init], [E_init], [D_init]


    for i in range(n_cycle):
        S_o_j = n_d*D_e[i]
        S_o.append(S_o_j)

        gamma_j = S_o_j/P_o[i]
        gamma.append(gamma_j)

        delta_j = update_delta(gamma_j, beta=beta)
        delta.append(delta_j)

        n_a_j = 1/gamma_j - delta_j
        n_a.append(n_a_j)

        E_o_j = (n_dE**(i+1)) * E_o[0]
        E_o.append(E_o_j)

        eff_j = update_eff(n_a=n_a_j, E_o=E_o_j, S_o=S_o_j, delta_e=delta_e, k_C=k_C, n_d=n_d)
        eff.append(eff_j)
        # print('At cycle {}, PCR efficiency is {}'.format(i, eff_j))

        D_e_j = (eff_j + 1)*D_e[i]
        D_e.append(D_e_j)

        P_o_j = P_o[i] - eff_j * S_o_j
        P_o.append(P_o_j)
    
    return eff, n_a, D_e, S_o, E_o, P_o, gamma, delta

In [34]:
# define constants
k_C = Decimal(15)
n_d = Decimal(1)
n_dE = Decimal(0.99)

beta = Decimal(23)

P_init = Decimal(9.e5)
E_init = Decimal(1e5)
delta_e = Decimal(50)


In [64]:
D_inits = []
D_es = []

for i in np.linspace(-6, 6, 10000):
    D_e = simulate_vals(D_init=Decimal(10.0**i), 
                        E_init=E_init, 
                        P_init=P_init, 
                        delta_e=delta_e,
                        n_d=n_d,
                        n_dE=n_dE,
                        k_C=k_C,
                        beta=beta,
                        n_cycle=49)[2]

    # plt.plot(D_e, label = 'modelled curve')
    # plt.legend()
    # plt.grid()

    D_inits.append(np.array([10.0**i]))
    D_es.append(np.array([float(str(x)) for x in D_e]))
        


curves_dict = {'D Init': D_inits, 'D Values': D_es}
curves_df = pd.DataFrame(curves_dict)

In [65]:
print(curves_df.head())
# curves_df.to_csv('data/curves_vary_D_init_params.csv', index=False)

                     D Init                                           D Values
0                   [1e-06]  [1e-06, 1.999999999987222e-06, 3.9999999999233...
1  [1.0027672000990755e-06]  [1.0027672000990755e-06, 2.0055344001853024e-0...
2  [1.0055420575945396e-06]  [1.0055420575945396e-06, 2.0110841151761593e-0...
3  [1.0083245936759398e-06]  [1.0083245936759398e-06, 2.0166491873388883e-0...
4  [1.0111148295914603e-06]  [1.0111148295914603e-06, 2.0222296591698572e-0...


In [70]:
import torch
from torch.utils.data import Dataset

class SynthCurveDataset(Dataset):
    def __init__(self, inputs, outputs):
        self.inputs = inputs
        self.outputs = outputs

    def __getitem__(self, index):
        return torch.tensor(self.inputs[index]), torch.tensor(self.outputs[index])
    
    def __len__(self):
        return len(self.inputs)
    


In [72]:
from sklearn.model_selection import train_test_split

data_train, data_val, labels_train, labels_val = train_test_split(D_es, D_inits, test_size=0.2, random_state=42)
train_dataset = SynthCurveDataset(data_train, labels_train)
val_dataset = SynthCurveDataset(data_val, labels_val)

print(len(train_dataset))
print(len(val_dataset))
print(train_dataset[0])
print(val_dataset[0])

8000
2000
(tensor([ 127617.6053,  183710.1314,  248450.8229,  318784.9681,  391703.2589,
         464577.3603,  535287.6576,  602237.5476,  664313.0304,  720819.1226,
         771409.2347,  816015.6862,  854785.4389,  888022.9675,  916141.0030,
         939619.2041,  958970.4172,  974713.9520,  987355.1726,  997370.6395,
        1005198.0256, 1011230.0404, 1015811.6323, 1019239.7912, 1021765.3295,
        1023596.0861, 1024901.0681, 1025815.1153, 1026443.7464, 1026867.9163,
        1027148.4820, 1027330.2406, 1027445.4575, 1027516.8537, 1027560.0578,
        1027585.5615, 1027600.2312, 1027608.4436, 1027612.9126, 1027615.2737,
        1027616.4831, 1027617.0828, 1027617.3704, 1027617.5034, 1027617.5628,
        1027617.5883, 1027617.5988, 1027617.6029, 1027617.6045, 1027617.6050],
       dtype=torch.float64), tensor([127617.6053], dtype=torch.float64))
(tensor([3.1853e+01, 6.3693e+01, 1.2733e+02, 2.5446e+02, 5.0810e+02, 1.0129e+03,
        2.0129e+03, 3.9757e+03, 7.7611e+03, 1.4834e+04

In [77]:
from resnet import EKGResNetModel
from base import fit_model
from torch.utils.data import DataLoader

NUM_EPOCHS = 20
batch_size = 50

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

example_data, example_label = train_dataset[0]

if len(example_data.shape) == 1:
    n_samples = example_data.shape[0]
    n_channels = 1
else:
    n_channels, n_samples = example_data.shape

n_outputs = example_label.shape[0]
regress = [True]*n_outputs

print("cuda available: " + str(torch.cuda.is_available()))

model = EKGResNetModel(n_channels=n_channels, n_samples=n_samples, n_outputs=n_outputs, num_rep_blocks=8, kernel_size=16, regress=[True]) #original num_rep_blocks 32
fit_model(model, train_dataloader, val_dataloader, save_path="output/resnet_lr1e-5", max_epochs=NUM_EPOCHS, learning_rate=1e-5)

cuda available: False

SEQ_LEN:  6 
OUT_FEATURES:  64 

net thinks it will have
  seq_len     : 6
  last active : 384
-------------------
fitting model:  {'save_path': 'output/resnet_lr1e-5', 'max_epochs': 20, 'learning_rate': 1e-05}
Enumerating batches, (epoch 0)


  0%|          | 0/160 [00:00<?, ?it/s]

RuntimeError: Given groups=1, weight of size [64, 1, 16], expected input[1, 50, 50] to have 1 channels, but got 50 channels instead